## 3: Feature engineering

In [1]:
import pandas as pd      
import geopandas as gpd   
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')


### Paths

In [2]:

script_dir=os.path.dirname(os.path.abspath('__file__'))
project_dir=os.path.dirname(script_dir)
data_dir=os.path.join(project_dir, 'Data')

crime_path=os.path.join(data_dir, 'crime_monthly_lsoa.csv')
weather_path=os.path.join(data_dir, 'weather_monthly.csv')
holidays_path=os.path.join(data_dir, 'School_and_bank_holidays.csv')
iod_path=os.path.join(data_dir, 'IoD-2025-custom_data_download-LSOA.csv')
shapefile_path=os.path.join(data_dir, 'shapefile', 'LSOA_2021_EW_BGC_V5.shp')
output_path=os.path.join(data_dir, 'crime_features_model.csv')


### Loading all data sources:

In [3]:

crime=pd.read_csv(crime_path)
weather=pd.read_csv(weather_path)
holidays=pd.read_csv(holidays_path, encoding='utf-8-sig')
holidays['date']=pd.to_datetime(holidays['date'], dayfirst=True)
iod=pd.read_csv(iod_path, encoding='utf-8-sig')
gdf=gpd.read_file(shapefile_path)

#Filtering shapefile to Teesside LSOAs
teesside_codes=crime['LSOA code'].unique()
teesside_gdf=gdf[gdf['LSOA21CD'].isin(teesside_codes)].copy().reset_index(drop=True)

print(f"Crime: {len(crime):,} rows | Weather: {len(weather)} months |"
      f"Holidays: {len(holidays)} days | IoD: {len(iod)} LSOAs | "
      f"Shapefile: {len(teesside_gdf)} LSOAs")


Crime: 13,284 rows | Weather: 36 months |Holidays: 1186 days | IoD: 359 LSOAs | Shapefile: 359 LSOAs


### Step 1: DEfining hotspot labels

In [4]:

print(f"Step 1: Defining hotspot labels")

#Counting unique LSOAs
n_lsoas=crime['LSOA code'].nunique()
#Setting hotspot treshold at 10%
top_pct=0.10
n_hotspot=max(1, int(n_lsoas*top_pct))

crime['hotspot']=0

#Looping through each month to see which 36 LSOAs had the most crime in that specific month
for month in crime['Month'].unique():
    mask=crime['Month']==month
    top_codes=crime.loc[mask].nlargest(n_hotspot, 'crime_count')['LSOA code'].values
    #Setting hotspot=1 (True) for the specific LSOA in that specific month
    crime.loc[mask&crime['LSOA code'].isin(top_codes), 'hotspot']=1

print(f"Hotspot threshold: top {int(top_pct*100)}% = {n_hotspot} LSOAs per month")
print(f"Hotspot=1: {crime['hotspot'].sum():,} ({crime['hotspot'].mean()*100:.1f}%)")

Step 1: Defining hotspot labels
Hotspot threshold: top 10% = 36 LSOAs per month
Hotspot=1: 1,296 (9.8%)


### Step 2: Lagged crime features

In [5]:
print(f"Step 2: Lagged crime features")

crime=crime.sort_values(['LSOA code', 'Month']).reset_index(drop=True)

#Lagged crime counts (1,2,3 motnhs back)
for lag in [1,2,3]:
    crime[f'crime_lag_{lag}']=crime.groupby('LSOA code')['crime_count'].shift(lag)

#Month-on-month change
crime['crime_change']=crime['crime_count']-crime['crime_lag_1']

#3-month rolling average (using previous month only)
crime['crime_rolling_3']=crime.groupby('LSOA code')['crime_count'].transform(
    lambda x:x.shift(1).rolling(3, min_periods=1).mean()
)

#Previous month's hotspot status
crime['hotspot_lag_1']=crime.groupby('LSOA code')['hotspot'].shift(1)


Step 2: Lagged crime features


### Step 3: Spatial lag (from neighbouring LSOAs)

In [6]:
print(f"STEP 3: Spatial lag")

from libpysal.weights import Queen

#Queen contiguity: areas are neighbours if boundaries touch (even at a corner)
w=Queen.from_dataframe(teesside_gdf, ids=teesside_gdf['LSOA21CD'].tolist())
print(f"Adjacemcy matrix: {w.n} LSOAs, mean {w.mean_neighbors:.1f} neighbours")

spatial_rows=[]
#Set of LSOA codes present in the shape file, only these have adjacency data
spatial_lsoas=set(teesside_gdf['LSOA21CD'])

for month in sorted(crime['Month'].unique()):
    #Indexing by LSOA code to look up neighbours values directly
    md=crime[crime['Month']==month].set_index('LSOA code')

    for lsoa in spatial_lsoas:
        if lsoa not in md.index:
            continue
        #Keeping only neighbours that exist in this month's data
        neighbours=[nb for nb in w.neighbors.get(lsoa, []) if nb in md.index]

        if neighbours:
            #Average crime and hotspot rate across surrounding areas this month
            avg_crime=np.mean([md.loc[nb, 'crime_count'] for nb in neighbours])
            avg_hotspot=np.mean([md.loc[nb, 'hotspot'] for nb in neighbours])

        else: 
            avg_crime, avg_hotspot=np.nan,0

        spatial_rows.append({'Month': month,
                             'LSOA code': lsoa,
                             'neighbour_avg_crime': avg_crime,
                             'neighbour_hotspot_pct': avg_hotspot})

#Sorting by LSOA code then Month, needed before shift()
sdf=pd.DataFrame(spatial_rows).sort_values(['LSOA code', 'Month'])

#Shifting by 1 to replace each row with the previous month values --> the model
# only sees neighbourhood context available at prediction time

sdf['spatial_lag_crime']=sdf.groupby('LSOA code')['neighbour_avg_crime'].shift(1)
sdf['spatial_lag_hotspot']=sdf.groupby('LSOA code')['neighbour_hotspot_pct'].shift(1)

#Merging back
crime=crime.merge(
    sdf[['Month', 'LSOA code', 'spatial_lag_crime', 'spatial_lag_hotspot']],
    on=['Month', 'LSOA code'], how='left'
)

STEP 3: Spatial lag
Adjacemcy matrix: 359 LSOAs, mean 5.8 neighbours


### Step 4: Temporal features

In [7]:
print(f"Step 4: Temporal features")

crime['Month_dt']=pd.to_datetime(crime['Month'])
crime['month_num']=crime['Month_dt'].dt.month
crime['year']=crime['Month_dt'].dt.year

#Cyclical encoding --> Dec and Jan are close)
crime['month_sin']=np.sin(2*np.pi*crime['month_num']/12)
crime['month_cos']=np.cos(2*np.pi*crime['month_num']/12)

print(f"Done: month_num, eyar, month_sin, mont_cos")




Step 4: Temporal features
Done: month_num, eyar, month_sin, mont_cos


### Step 5: Weather data features

In [8]:
print(f"STEP 5: Weather features")

crime=crime.merge(weather, on='Month', how="left")
#Chcking for missing values
weather_missing=crime['temp_mean'].isna().sum()
if weather_missing>0:
    print(f"{weather_missing} rows missing weather data")
    
print(f"Done: temp_mean, temp_max, temp_min, rainfall_total, rainfall_days")

STEP 5: Weather features
Done: temp_mean, temp_max, temp_min, rainfall_total, rainfall_days


### Step 6: School & Bank holiday features

In [9]:
print(f"STEP 6: School & Bank holiday features")

holidays['Month']=holidays['date'].dt.to_period('M').astype(str)

#Grouping the holidays by month
holiday_monthly=holidays.groupby('Month').agg(
    school_holiday_days=('school_holiday', 'sum'),
    bank_holiday_days=('bank_holiday', 'sum')
).reset_index()

crime=crime.merge(holiday_monthly, on='Month', how="left")

print(f"Done: school_holiday_days, bank_holiday_days")




STEP 6: School & Bank holiday features
Done: school_holiday_days, bank_holiday_days


### Step 7: IoD 2025 Deprivation features

In [10]:

print(f"STEP 7: Deprivation features")

iod_rename={
    'LSOA code (2021)':'LSOA code',
    'Index of Multiple Deprivation (IMD) Decile':'imd_decile',
    'Income Decile':'income_decile',
    'Employment Decile':'employment_decile',
    'Education Skills and Training Decile':'education_decile',
    'Health Deprivation and Disability Decile':'health_decile',
    'Barriers to Housing and Services Decile':'housing_decile',
    'Living Environment Decile':'living_env_decile'
}

#Renaming original IoD columns
iod_features=iod.rename(columns=iod_rename)[list(iod_rename.values())]
crime=crime.merge(iod_features, on='LSOA code', how='left')

#Checking if IoD LSOAs and crime LSOAs match
unmatched=crime[crime['imd_decile'].isna()]['LSOA code'].nunique()

print(f"Matched LSOAs: {crime['imd_decile'].notna().sum():,} rows")
print(f"Unmatched LSOAs: {unmatched} (post-2021 boundary codes)")

STEP 7: Deprivation features
Matched LSOAs: 12,924 rows
Unmatched LSOAs: 10 (post-2021 boundary codes)


### Step 8: Local Authority One-hot encoding

In [11]:
print(f"STEP 8: Local authority encoding:")

la_dummies=pd.get_dummies(crime['Local_Authority'], prefix='la', dtype=int)
crime=pd.concat([crime, la_dummies], axis=1)

print(f"Done: {list(la_dummies.columns)}")

STEP 8: Local authority encoding:
Done: ['la_Hartlepool', 'la_Middlesbrough', 'la_Redcar and Cleveland', 'la_Stockton-on-Tees']


### Step 9: Handling missing values

In [12]:
print(f"STEP 9: Handling missing values")

print(f"Rows before: {len(crime):,}")

#Dropping rows  where lag features are NaN (first 3 months of each LSOA)
crime_model=crime.dropna(subset=['crime_lag_1', 'crime_lag_2', 'crime_lag_3', 'spatial_lag_crime', 'hotspot_lag_1']).copy()

print(f"ROws after dropping NaN lags: {len(crime_model):,}")
print(f"Months: {crime_model['Month'].nunique()}"
      f"({crime_model['Month'].min()} to {crime_model['Month'].max()})")


STEP 9: Handling missing values
Rows before: 13,284
ROws after dropping NaN lags: 11,847
Months: 33(2023-07 to 2026-03)


### Step 10: Defining features and saving

In [13]:

feature_cols=[
    #Lagged crime features
    'crime_lag_1', 'crime_lag_2', 'crime_lag_3', 'crime_change',
    'crime_rolling_3', 'hotspot_lag_1',
    #SPatial features
    'spatial_lag_crime', 'spatial_lag_hotspot',
    # Temporal features
    'month_sin', 'month_cos',
    #Weather features
    'temp_mean', 'temp_max', 'temp_min', 'rainfall_total', 'rainfall_days',
    #Holiday features
    'school_holiday_days', 'bank_holiday_days',
    #Deprivation features
    'imd_decile','income_decile','employment_decile', 'education_decile',
    'health_decile', 'housing_decile', 'living_env_decile',
    #Local Authority
    'la_Hartlepool', 'la_Middlesbrough', 'la_Redcar and Cleveland','la_Stockton-on-Tees' 
]

#Final Nan check
nan_warnings=[]
for col in feature_cols:
    n=crime_model[col].isna().sum()
    if n>0:
        nan_warnings.append(f"{col} has {n} Nan values")

if nan_warnings:
    for w in nan_warnings:
        print(w)

#SAVING
model_cols=(['Month', 'LSOA code', 'LSOA name', 'Local_Authority', 'crime_count']
            + feature_cols + ['hotspot'])

crime_model[model_cols].to_csv(output_path, index=False)


### Summary

In [14]:
print(f"Feature Engineering complete")
print(f"Final dataset: {len(crime_model):,} rows x {len(feature_cols)} features")
print(f"LSOAs: {crime_model['LSOA code'].nunique()}")
print(f"Months: {crime_model['Month'].nunique()} "
      f"({crime_model['Month'].min()} to {crime_model['Month'].max()})")
print(f"\nTarget balance:")
print("-"*40)
print(f"Hotspot=1: {crime_model['hotspot'].sum():,} ({crime_model['hotspot'].mean()*100:.1f}%)")
print(f"Hotspot=0: {(crime_model['hotspot']==0).sum():,} "
      f"({(crime_model['hotspot']==0).mean()*100:.1f}%)")
print(f"\nFeatures ({len(feature_cols)}):")
print("-"*40)
for i, col in enumerate(feature_cols, 1):
    print(f" {i:2d}. {col}")
print(f"Saved: {output_path}")

Feature Engineering complete
Final dataset: 11,847 rows x 28 features
LSOAs: 359
Months: 33 (2023-07 to 2026-03)

Target balance:
----------------------------------------
Hotspot=1: 1,188 (10.0%)
Hotspot=0: 10,659 (90.0%)

Features (28):
----------------------------------------
  1. crime_lag_1
  2. crime_lag_2
  3. crime_lag_3
  4. crime_change
  5. crime_rolling_3
  6. hotspot_lag_1
  7. spatial_lag_crime
  8. spatial_lag_hotspot
  9. month_sin
 10. month_cos
 11. temp_mean
 12. temp_max
 13. temp_min
 14. rainfall_total
 15. rainfall_days
 16. school_holiday_days
 17. bank_holiday_days
 18. imd_decile
 19. income_decile
 20. employment_decile
 21. education_decile
 22. health_decile
 23. housing_decile
 24. living_env_decile
 25. la_Hartlepool
 26. la_Middlesbrough
 27. la_Redcar and Cleveland
 28. la_Stockton-on-Tees
Saved: c:\Users\marco\Documents\Masters Project\Data\crime_features_model.csv
